# Visual and Presentable 

## Export all dashboard data:

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('search_experiment.csv')
control   = df[df['group'] == 'control']
treatment = df[df['group'] == 'treatment']

# ── 1. KPI Summary ───────────────────────────────────────────
kpi = pd.DataFrame({
    'metric'   : ['CTR (%)', 'Avg Dwell Time (sec)', 'Bounce Rate (%)'],
    'control'  : [
        round(control['clicked'].mean() * 100, 2),
        round(control[control['clicked']==1]['dwell_time_sec'].mean(), 2),
        round(control['bounced'].mean() * 100, 2)
    ],
    'treatment': [
        round(treatment['clicked'].mean() * 100, 2),
        round(treatment[treatment['clicked']==1]['dwell_time_sec'].mean(), 2),
        round(treatment['bounced'].mean() * 100, 2)
    ]
})
kpi['lift'] = (kpi['treatment'] - kpi['control']).round(2)
kpi.to_csv('pbi_kpi_summary.csv', index=False)
print("KPI Summary:")
print(kpi)
print()

# ── 2. Daily CTR Trend ───────────────────────────────────────
daily = df.groupby(['day','group'])['clicked'].mean().mul(100).round(2).reset_index()
daily.columns = ['day', 'group', 'ctr']
daily.to_csv('pbi_daily_trend.csv', index=False)
print("Daily trend exported.")

# ── 3. Click Position Distribution ──────────────────────────
pos = df[df['click_position'].notna()].groupby(['group','click_position']).size().reset_index(name='count')
pos_total = pos.groupby('group')['count'].transform('sum')
pos['pct'] = (pos['count'] / pos_total * 100).round(2)
pos.to_csv('pbi_click_position.csv', index=False)
print("Click position exported.")

# ── 4. Confidence Interval ───────────────────────────────────
n_ctrl  = len(control)
n_treat = len(treatment)
p_ctrl  = control['clicked'].mean()
p_treat = treatment['clicked'].mean()
diff    = p_treat - p_ctrl
se      = np.sqrt((p_ctrl*(1-p_ctrl)/n_ctrl) + (p_treat*(1-p_treat)/n_treat))
z       = 1.96

ci = pd.DataFrame({
    'label'      : ['Lower Bound', 'Observed Lift', 'Upper Bound'],
    'ctr_lift_pct': [
        round((diff - z*se)*100, 2),
        round(diff*100, 2),
        round((diff + z*se)*100, 2)
    ]
})
ci.to_csv('pbi_confidence_interval.csv', index=False)
print("Confidence interval exported.")

# ── 5. Weekly Summary ────────────────────────────────────────
df['week'] = pd.cut(df['day'], bins=[0,7,14,21], labels=['Week 1','Week 2','Week 3'])
weekly = df.groupby(['week','group'])['clicked'].mean().mul(100).round(2).reset_index()
weekly.columns = ['week','group','ctr']
weekly.to_csv('pbi_weekly_trend.csv', index=False)
print("Weekly trend exported.")

print()
print("All 5 CSVs exported and ready for Power BI!")

KPI Summary:
                 metric  control  treatment   lift
0               CTR (%)    48.48      54.89   6.41
1  Avg Dwell Time (sec)   151.29     180.71  29.42
2       Bounce Rate (%)    34.88      25.17  -9.71

Daily trend exported.
Click position exported.
Confidence interval exported.
Weekly trend exported.

All 5 CSVs exported and ready for Power BI!


In [6]:
import os

# Check current folder
print("Current folder:", os.getcwd())

# List all CSV files available
for f in os.listdir():
    if f.endswith('.csv'):
        print("Found:", f)

        import pandas as pd
import numpy as np
from scipy import stats
import os

# Load the data
df = pd.read_csv('search_experiment.csv')
control   = df[df['group'] == 'control']
treatment = df[df['group'] == 'treatment']

# KPI Summary
kpi = pd.DataFrame({
    'metric'   : ['CTR (%)', 'Avg Dwell Time (sec)', 'Bounce Rate (%)'],
    'control'  : [
        round(control['clicked'].mean() * 100, 2),
        round(control[control['clicked']==1]['dwell_time_sec'].mean(), 2),
        round(control['bounced'].mean() * 100, 2)
    ],
    'treatment': [
        round(treatment['clicked'].mean() * 100, 2),
        round(treatment[treatment['clicked']==1]['dwell_time_sec'].mean(), 2),
        round(treatment['bounced'].mean() * 100, 2)
    ]
})
kpi['lift'] = (kpi['treatment'] - kpi['control']).round(2)
kpi.to_csv('pbi_kpi_summary.csv', index=False)

# Daily trend
daily = df.groupby(['day','group'])['clicked'].mean().mul(100).round(2).reset_index()
daily.columns = ['day', 'group', 'ctr']
daily.to_csv('pbi_daily_trend.csv', index=False)

# Click position
pos = df[df['click_position'].notna()].groupby(['group','click_position']).size().reset_index(name='count')
pos_total = pos.groupby('group')['count'].transform('sum')
pos['pct'] = (pos['count'] / pos_total * 100).round(2)
pos.to_csv('pbi_click_position.csv', index=False)

# Confidence interval
n_ctrl  = len(control)
n_treat = len(treatment)
p_ctrl  = control['clicked'].mean()
p_treat = treatment['clicked'].mean()
diff    = p_treat - p_ctrl
se      = np.sqrt((p_ctrl*(1-p_ctrl)/n_ctrl) + (p_treat*(1-p_treat)/n_treat))
z       = 1.96
ci = pd.DataFrame({
    'label'       : ['Lower Bound', 'Observed Lift', 'Upper Bound'],
    'ctr_lift_pct': [round((diff - z*se)*100, 2), round(diff*100, 2), round((diff + z*se)*100, 2)]
})
ci.to_csv('pbi_confidence_interval.csv', index=False)

# Weekly trend
df['week'] = pd.cut(df['day'], bins=[0,7,14,21], labels=['Week 1','Week 2','Week 3'])
weekly = df.groupby(['week','group'])['clicked'].mean().mul(100).round(2).reset_index()
weekly.columns = ['week','group','ctr']
weekly.to_csv('pbi_weekly_trend.csv', index=False)

# 3 individual card CSVs
for _, row in kpi.iterrows():
    name = row['metric'].replace(' ','_').replace('(','').replace(')','').replace('%','pct')
    pd.DataFrame({
        'control'  : [row['control']],
        'treatment': [row['treatment']],
        'lift'     : [row['lift']]
    }).to_csv(f'pbi_card_{name}.csv', index=False)

print("All files created in:", os.getcwd())
for f in os.listdir():
    if f.startswith('pbi_'):
        print(" ", f)

Current folder: C:\Users\chava
Found: baseline_model_results.csv
Found: brfss_smart_2013_2023.csv
Found: result_bounce.csv
Found: result_ctr.csv
Found: result_daily_trend.csv
Found: result_dwell.csv
Found: result_positions.csv
Found: result_weekly_ctr.csv
Found: search_experiment.csv
All files created in: C:\Users\chava
  pbi_card_Avg_Dwell_Time_sec.csv
  pbi_card_Bounce_Rate_pct.csv
  pbi_card_CTR_pct.csv
  pbi_click_position.csv
  pbi_confidence_interval.csv
  pbi_daily_trend.csv
  pbi_kpi_summary.csv
  pbi_weekly_trend.csv


In [5]:
import os
print(os.getcwd())

C:\Users\chava


In [7]:
# Create 3 separate simple card CSVs
import pandas as pd

kpi = pd.read_csv('pbi_kpi_summary.csv')

for _, row in kpi.iterrows():
    name = row['metric'].replace(' ','_').replace('(','').replace(')','').replace('%','pct')
    pd.DataFrame({
        'control'  : [row['control']],
        'treatment': [row['treatment']],
        'lift'     : [row['lift']]
    }).to_csv(f'pbi_card_{name}.csv', index=False)

print("3 card CSVs created!")

3 card CSVs created!
